# Module 08: Token Matching


# 8.1 Introduction to the Matcher


## 🔍 Why Rule-Based Matching?

Statistical models (like the NER we used earlier) are great when you want to generalize (e.g., finding all people's names without knowing them beforehand).

But sometimes, you want to find **exact patterns** that always follow a specific rule (e.g., finding phone numbers, IP addresses, or very specific phrases).

Unlike Regular Expressions (RegEx) which only look at raw strings, spaCy's `Matcher` lets you look for patterns in the **Token objects** and their linguistic annotations (like POS tags and lemmas)!


## 🛠️ The `Matcher` Class

The Matcher requires the `vocab` of the `nlp` object so it can use the same `StringStore` hashes.


In [1]:
import spacy
from spacy.matcher import Matcher

nlp = spacy.load("en_core_web_sm")

# Initialize the matcher with the shared vocabulary
matcher = Matcher(nlp.vocab)


## 🧩 Creating a Pattern

A pattern is a list of dictionaries. **Each dictionary represents ONE token.**

Let's build a pattern to find the exact phrase "hello world".
- Token 1 must be "hello" (case-insensitive).
- Token 2 must be "world" (case-insensitive).


In [2]:
# The pattern: two tokens
pattern = [{"LOWER": "hello"}, {"LOWER": "world"}]

# Add the pattern to the matcher under the ID 'HelloWorld'
matcher.add("HelloWorld", [pattern])

doc = nlp("I said hello world to my computer. Then Hello World again!")

# Calling the matcher on the doc returns a list of matches
matches = matcher(doc)

print(f"Found {len(matches)} matches:\n")
for match_id, start, end in matches:
    # Get the string representation of the match_id hash
    string_id = nlp.vocab.strings[match_id]  
    # Slice the document to get the matched span
    matched_span = doc[start:end]      
    print(f"Match ID: {string_id} | Span: '{matched_span.text}' | Indices: {start} to {end}")


Found 2 matches:

Match ID: HelloWorld | Span: 'hello world' | Indices: 2 to 4
Match ID: HelloWorld | Span: 'Hello World' | Indices: 9 to 11



<br><br>

---

<br><br>


# 8.2 Pattern Syntax and Wildcards


## 📜 Syntax Rules

As you saw, a pattern is a list of dictionaries. 
```python
[{"ATTRIBUTE": "VALUE"}, {"ATTRIBUTE": "VALUE"}]
```

But you can have multiple conditions for a SINGLE token by adding more keys to the dictionary!

Let's find a token that has the lemma "love" AND is a Verb, followed by a token that is a Noun.


In [3]:
import spacy
from spacy.matcher import Matcher

nlp = spacy.load("en_core_web_sm")
matcher = Matcher(nlp.vocab)

pattern = [
    {"LEMMA": "love", "POS": "VERB"},  # Token 1
    {"POS": "NOUN"}                    # Token 2
]

matcher.add("LoveNoun", [pattern])

doc = nlp("I love dogs. She loved cats. He loves coding. I have a love letter.")
matches = matcher(doc)

for match_id, start, end in matches:
    print(f"Match: {doc[start:end]}")


Match: love dogs
Match: loved cats


Notice how it missed "love letter"? That's because "love" in that context is acting as a noun modifier, not a verb! Our linguistic rules made the match much smarter than standard RegEx.

## 🃏 Wildcards

What if you want to find a specific word, then *any* word, then another specific word?
You can use an empty dictionary `{}` to represent a wildcard token!


In [4]:
matcher2 = Matcher(nlp.vocab)

# Match: "Data" -> ANY TOKEN -> "Science"
pattern_wildcard = [
    {"LOWER": "data"},
    {},  # Matches any single token
    {"LOWER": "science"}
]

matcher2.add("DataScience", [pattern_wildcard])

doc2 = nlp("He studies Data and Science. Data for Science is great. Data Science is not matched!")

print("Wildcard Matches:")
for match_id, start, end in matcher2(doc2):
    print(f"- {doc2[start:end]}")


Wildcard Matches:
- Data and Science
- Data for Science



<br><br>

---

<br><br>


# 8.3 Quantifiers (Operators)


## 🔢 Operators (OP)

In standard RegEx, you use symbols like `?`, `*`, and `+` to denote how many times a pattern should repeat. 

In spaCy's Matcher, you do this by adding the `"OP"` key to a token dictionary.

| OP | Description |
|----|-------------|
| `!` | Negate the pattern (match 0 times) |
| `?` | Optional (match 0 or 1 times) |
| `+` | Match 1 or more times |
| `*` | Match 0 or more times |


In [5]:
import spacy
from spacy.matcher import Matcher

nlp = spacy.load("en_core_web_sm")
matcher = Matcher(nlp.vocab)

# Let's match: "buy" (verb) + optional determiner ("a", "the") + noun
pattern = [
    {"LEMMA": "buy"},                   # Token 1: Base form 'buy'
    {"POS": "DET", "OP": "?"},          # Token 2: Optional determiner
    {"POS": "NOUN"}                     # Token 3: A Noun
]

matcher.add("BuyNoun", [pattern])

doc = nlp("I bought a car. She buys computers. They bought the house.")

for match_id, start, end in matcher(doc):
    print(f"Match: {doc[start:end]}")


Match: bought a car
Match: buys computers
Match: bought the house


## ⚠️ The Danger of `*` and `+`

Be very careful when using `"OP": "*"` (0 or more). Because it can match 0 tokens, it can sometimes match sequences you don't expect, or create combinatorial explosions (matching every possible sub-slice of a long phrase).


In [6]:
matcher2 = Matcher(nlp.vocab)

# Let's match a sequence of one or more numbers
pattern_nums = [
    {"IS_DIGIT": True, "OP": "+"}
]

matcher2.add("Numbers", [pattern_nums])

doc2 = nlp("My pin is 1 2 3 4.")

print("Notice how it finds every possible combination of the numbers!")
for match_id, start, end in matcher2(doc2):
    print(f"Match: {doc2[start:end]}")


Notice how it finds every possible combination of the numbers!
Match: 1
Match: 1 2
Match: 2
Match: 1 2 3
Match: 2 3
Match: 3
Match: 1 2 3 4
Match: 2 3 4
Match: 3 4
Match: 4



<br><br>

---

<br><br>


# 8.4 Available Pattern Attributes


## 🎛️ What can we match on?

You can match on almost every property of the `Token` object. 

Here are the most common attributes used in the Matcher:

### Exact Text / Lexical
- `ORTH`: The exact text of the token (case-sensitive).
- `LOWER`: The lowercase text.
- `LENGTH`: The length of the string.

### Rules
- `IS_ALPHA`, `IS_ASCII`, `IS_DIGIT`
- `IS_LOWER`, `IS_UPPER`, `IS_TITLE`
- `IS_PUNCT`, `IS_SPACE`, `IS_STOP`
- `LIKE_NUM`, `LIKE_URL`, `LIKE_EMAIL`

### Linguistic Predictions
- `POS`, `TAG`, `DEP`, `LEMMA`
- `ENT_TYPE`: The Named Entity label of the token.

Let's combine several of these to find a complex pattern: **An email address, followed by an optional space, followed by a Verb.**


In [7]:
import spacy
from spacy.matcher import Matcher

nlp = spacy.load("en_core_web_sm")
matcher = Matcher(nlp.vocab)

pattern = [
    {"LIKE_EMAIL": True},            # Token 1: An email
    {"IS_SPACE": True, "OP": "?"},   # Token 2: Optional space
    {"POS": "VERB"}                  # Token 3: A verb
]

matcher.add("EmailAction", [pattern])

doc = nlp("Please ensure test@example.com receives the file. Also user@spacy.io  bounced yesterday.")

for match_id, start, end in matcher(doc):
    print(f"Match: {doc[start:end]}")


Match: test@example.com receives
Match: user@spacy.io  bounced


## 📏 Matching using `IN`

What if you want to match a token if it's in a specific list of words? Use the `IN` operator!
```python
{"LOWER": {"IN": ["apple", "banana", "orange"]}}
```

Let's find occurrences of fruits followed by the word "juice".


In [8]:
matcher2 = Matcher(nlp.vocab)

pattern_fruit = [
    {"LOWER": {"IN": ["apple", "orange", "grape"]}},
    {"LOWER": "juice"}
]

matcher2.add("FruitJuice", [pattern_fruit])

doc2 = nlp("I drank some orange juice. He spilled his apple juice.")
for match_id, start, end in matcher2(doc2):
    print(f"Match: {doc2[start:end]}")


Match: orange juice
Match: apple juice



<br><br>

---

<br><br>


# 8.5 On-Match Callbacks


## 📞 What is a Callback?

When `matcher.add()` is called, the second argument (which we have been leaving blank or avoiding by passing lists directly) is an `on_match` callback function.

If you provide a function, spaCy will automatically execute that function **every time it finds a match**.

This is incredibly useful if you want to mutate the document automatically (like merging the matched tokens together) or trigger an external API!


In [9]:
import spacy
from spacy.matcher import Matcher

nlp = spacy.load("en_core_web_sm")
matcher = Matcher(nlp.vocab)

# Define the callback function
# It MUST accept exactly these 4 arguments
def on_match_found(matcher, doc, id, matches):
    match_id, start, end = matches[id]
    span = doc[start:end]
    print(f"[CALLBACK TRIGGERED] Found matching span: '{span.text}'")

# Let's find U.S. states followed by 'state'
pattern = [
    {"LOWER": {"IN": ["texas", "california", "florida"]}},
    {"LOWER": "state"}
]

# We pass the callback function as the second argument
matcher.add("StateMatch", [pattern], on_match=on_match_found)

doc = nlp("I live in the California state. He moved to the Texas state.")

print("Processing document...")
# The matcher will print from inside the callback!
matches = matcher(doc)
print("Processing complete.")


Processing document...
[CALLBACK TRIGGERED] Found matching span: 'California state'
[CALLBACK TRIGGERED] Found matching span: 'Texas state'
Processing complete.



<br><br>

---

<br><br>


# 8.6 Complex Patterns & Debugging


## 🧠 Combining Rules for Complex Extraction

Let's put everything we've learned together to build a robust pattern that extracts monetary values combined with directional words (e.g., "increased by $5 million").

Our rules:
1. A verb indicating direction (increase, decrease, rise, fall, drop)
2. An optional preposition ("by", "to")
3. An entity that is classified as MONEY by the NER model.
   *Wait, how do we match a multi-token entity like "$5 million" if the matcher evaluates one token at a time?*
   *Answer: We match any token whose `ENT_TYPE` is "MONEY", and we use the `+` operator!*


In [10]:
import spacy
from spacy.matcher import Matcher

nlp = spacy.load("en_core_web_sm")
matcher = Matcher(nlp.vocab)

directional_verbs = ["increase", "decrease", "rise", "fall", "drop", "jump"]

pattern = [
    {"LEMMA": {"IN": directional_verbs}},       # 1. The directional verb
    {"POS": "ADP", "OP": "?"},                  # 2. Optional preposition (ADP)
    {"ENT_TYPE": "MONEY", "OP": "+"}            # 3. One or more tokens that are part of a MONEY entity
]

matcher.add("MoneyMovement", [pattern])

doc = nlp("Revenues increased by $5 million. Costs dropped to €2. The stock jumped $500 today.")

for match_id, start, end in matcher(doc):
    print(f"Match: {doc[start:end]}")


Match: increased by $
Match: increased by $5
Match: increased by $5 million


Notice how it caught all three variations perfectly, adapting to different lengths of monetary values and the presence or absence of prepositions!

## 🐞 Debugging Patterns

When building complex patterns with multiple `OP` quantifiers, you might get overlapping matches or missed matches. To debug:

1. Start simple. Remove `OP` quantifiers and ensure the base words match.
2. Add quantifiers one by one.
3. Make sure you aren't relying on linguistic features (`POS`, `DEP`) that the model predicted incorrectly for your specific test text! If the NER model fails to tag "$5 million" as MONEY, the Matcher will also fail to find it.

## 🎉 Summary of Module 8

You've unlocked the power of rule-based NLP!
- You learned how to define token-level dictionary rules.
- You understand how to use `OP` quantifiers (`?`, `*`, `+`) and wildcards (`{}`).
- You know how to trigger custom python functions via `on_match` callbacks.

In **Module 9**, we will explore **Advanced Matching**, where we look at the blazing fast `PhraseMatcher` and the incredibly powerful syntactic `DependencyMatcher`.
